# Preserve a .o4d through USDC

Set the paths below. This works with every supported codec and needs OpenUSD, but no codec runtime.
The USDC stores compressed native state in a custom O4D prim; it is not a renderable mesh export.

In [ ]:
import hashlib
from pathlib import Path
import open4d
from open4d.codec import inspect_o4d, pack_o4d, unpack_o4d

source = Path("motion.o4d")
output = Path("output/native-usdc")

In [ ]:
info = inspect_o4d(source)
print(info["codec"], info["frame_count"], "frames")

In [ ]:
with open4d.NativeSequence(source) as native:
    open4d.save(native, output / "native.usdc", overwrite=True)

In [ ]:
with open4d.load(output / "native.usdc") as native:
    restored = open4d.save(native, output / "restored.o4d", overwrite=True)

with source.open("rb") as a, restored.open("rb") as b:
    assert hashlib.sha256(a.read()).digest() == hashlib.sha256(b.read()).digest()
print("Identical compressed bytes")

To extract and repack the same native payloads without decoding:

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as folder:
    native_files = unpack_o4d(source, Path(folder) / "native")
    repacked = pack_o4d(native_files, output / "repacked.o4d", overwrite=True)
print(inspect_o4d(repacked)["codec"])
